<a href="https://colab.research.google.com/github/Lopezbackend-devops/DO180-apps/blob/master/parte_a_cnn_trade_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de Fotos de Anaqueles con CNN

## Objetivo

Construir un modelo de inteligencia artificial capaz de clasificar imágenes de anaqueles en dos categorías:

* **Aprobadas**: Foto correcta del anaquel.
* **Rechazadas**: Foto incorrecta (mala iluminación, mala posición, desorden, etc).

Este modelo busca **automatizar la validación de fotos tomadas por mercaderistas**, reduciendo la carga de trabajo de los analistas.



In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn

In [ ]:
import torch
import torchvision

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

Dispositivo: cuda


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset (3).zip


In [ ]:
!unzip dataset.zip

Archive:  dataset.zip
replace aprobadas/IMG-20260309-WA0122.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: aprobadas/IMG-20260309-WA0122.jpg  
replace aprobadas/IMG-20260309-WA0125.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace aprobadas/IMG-20260309-WA0125.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: aprobadas/IMG-20260309-WA0125.jpg  
  inflating: aprobadas/IMG-20260309-WA0142.jpg  
  inflating: aprobadas/IMG-20260309-WA0143.jpg  
  inflating: aprobadas/IMG-20260309-WA0144.jpg  
  inflating: aprobadas/IMG-20260309-WA0145.jpg  
  inflating: aprobadas/IMG-20260309-WA0146.jpg  
  inflating: aprobadas/IMG-20260309-WA0147.jpg  
  inflating: aprobadas/IMG-20260309-WA0148.jpg  
  inflating: aprobadas/IMG-20260309-WA0149.jpg  
  inflating: rechazadas/IMG-20260309-WA0131.jpg  
  inflating: rechazadas/IMG-20260309-WA0132.jpg  
  inflating: rechazadas/IMG-20260309-WA0133.jpg  
  inflating: rechazadas/IMG-20260309-WA0134.jpg  
  inflat

In [ ]:
!ls dataset!ls dataset

ls: cannot access 'dataset!ls': No such file or directory
ls: cannot access 'dataset': No such file or directory


In [ ]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(brightness=0.2),

    transforms.ToTensor()

])

In [ ]:

dataset = datasets.ImageFolder(
    root="dataset",
    transform=transform
)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset'

In [ ]:
print(dataset.classes)

In [ ]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=4)

test_loader = DataLoader(test_dataset, batch_size=4)

In [ ]:
class ShelfCNN(nn.Module):

    def __init__(self):

        super(ShelfCNN, self).__init__()

        self.conv1 = nn.Conv2d(3,16,3)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16,32,3)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32,64,3)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(64*26*26,1)

    def forward(self,x):

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = torch.flatten(x,1)

        x = self.dropout(x)

        x = torch.sigmoid(self.fc(x))

        return x

In [ ]:
model = ShelfCNN().to(device)

In [ ]:
criterion = nn.BCELoss()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 20

train_losses = []

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_losses.append(total_loss)

    print("Epoch:", epoch, "Loss:", total_loss)

In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        preds = (outputs > 0.5).int()

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.numpy())

In [ ]:
cm = confusion_matrix(true_labels, predictions)

sns.heatmap(cm, annot=True, cmap="Blues")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(true_labels, predictions)

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)

plt.title("ROC Curve")

plt.show()

In [ ]:
plt.plot(train_losses)

plt.title("Training Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.show()

In [ ]:
print(classification_report(true_labels, predictions))

In [25]:
# Verificar falsos positivos

false_positives = 0

for i in range(len(true_labels)):

    if true_labels[i] == 1 and predictions[i] == 0:
        false_positives += 1

print("Cantidad de falsos positivos:", false_positives)

Cantidad de falsos positivos: 0


In [ ]:
torch.save(model.state_dict(), "modelo_cnn.pth")